# 第11章：现代 Transformer 架构改进

## 本章目标
- 将 GPT-2 架构升级到 2024-2025 年主流设计
- 理解并实现四大核心改进：**RMSNorm**、**SwiGLU**、**RoPE**、**GQA**
- 在 CPU 上用 baby 模型验证每个改进的效果

## 前置知识
- 第1章：GPT 架构（CausalSelfAttention、MLP、Block、GPT）
- PyTorch `nn.Module` 基本用法
- 矩阵运算和 softmax

## 改进路线图

| 组件 | GPT-2 (2019) | 现代 (Llama/Mistral/...) |
|------|-------------|------------------------|
| 归一化 | LayerNorm | **RMSNorm** |
| 激活函数 | GELU | **SwiGLU** |
| 位置编码 | 可学习绝对位置编码 | **RoPE**（旋转位置编码） |
| 注意力机制 | MHA（多头注意力） | **GQA**（分组查询注意力） |

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib
else:
    print("本地环境运行，请确保已安装 torch")

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

---

## 11.1 RMSNorm vs LayerNorm

### 理论

LayerNorm 的公式：

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma} \cdot \gamma + \beta$$

其中 $\mu$ 是均值，$\sigma$ 是标准差，$\gamma$ 和 $\beta$ 是可学习参数。

RMSNorm 的公式：

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot g, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

**核心区别**：RMSNorm 去掉了均值中心化（不减均值）和偏置项 $\beta$，只保留缩放因子 $g$。

**为什么有效**：
1. **计算更快**：不需要计算均值，减少了约 7-64% 的计算时间（Zhang & Sennrich, 2019）
2. **效果相当**：实验表明在大多数任务上与 LayerNorm 性能相当
3. **参数更少**：只有 $g$（无 $\beta$），每个隐藏维度省一个参数

Llama、Mistral、DeepSeek 等现代模型全部使用 RMSNorm。

参考论文：[Root Mean Square Layer Normalization](https://arxiv.org/abs/1910.07467) (Zhang & Sennrich, 2019)

In [ ]:
# ============================================================
# 手写 RMSNorm（~10 行核心逻辑）
# ============================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization
    
    与 LayerNorm 的区别：
    - 不做均值中心化（不减均值）
    - 没有偏置参数 beta
    - 只保留缩放参数 weight (g)
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # 可学习缩放因子 g

    def _norm(self, x):
        # RMS(x) = sqrt(mean(x^2) + eps)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

In [ ]:
# ============================================================
# 对比 LayerNorm vs RMSNorm：参数量、计算、输出差异
# ============================================================

torch.manual_seed(42)
dim = 128
x = torch.randn(2, 32, dim)  # (batch, seq_len, dim)

# 实例化两种归一化
ln = nn.LayerNorm(dim)
rn = RMSNorm(dim)

# 参数量对比
ln_params = sum(p.numel() for p in ln.parameters())  # weight + bias
rn_params = sum(p.numel() for p in rn.parameters())  # weight only
print(f"LayerNorm 参数量: {ln_params} (weight={dim}, bias={dim})")
print(f"RMSNorm  参数量: {rn_params} (weight={dim})")
print(f"参数节省: {dim} 个/层 ({100 * dim / ln_params:.0f}% 减少)")

# 输出对比
y_ln = ln(x)
y_rn = rn(x)

print(f"\nLayerNorm 输出均值: {y_ln.mean(dim=-1)[0, :5].tolist()}")
print(f"RMSNorm  输出均值: {y_rn.mean(dim=-1)[0, :5].tolist()}")
print(f"\nLayerNorm 输出标准差: {y_ln.std(dim=-1)[0, :5].tolist()}")
print(f"RMSNorm  输出标准差: {y_rn.std(dim=-1)[0, :5].tolist()}")

# 速度对比
n_iters = 10000
start = time.time()
for _ in range(n_iters):
    _ = ln(x)
ln_time = time.time() - start

start = time.time()
for _ in range(n_iters):
    _ = rn(x)
rn_time = time.time() - start

print(f"\nLayerNorm 耗时: {ln_time*1000:.1f} ms ({n_iters} 次)")
print(f"RMSNorm  耗时: {rn_time*1000:.1f} ms ({n_iters} 次)")
print(f"速度比: LayerNorm/RMSNorm = {ln_time/rn_time:.2f}x")

### 分析

- LayerNorm 会强制输出均值为 0、方差为 1（因为有均值中心化和标准差归一化）
- RMSNorm 只做尺度归一化（除以 RMS），不改变均值，但同样稳定了数值范围
- 参数量上 RMSNorm 少了一半（只有 weight，没有 bias）
- 对于大模型（如 Llama-70B 有 80 层），每层有 2 个归一化层，省下的参数和计算量相当可观
- 实际工程中，RMSNorm 配合 float16/bfloat16 训练也更稳定（在 `.float()` 精度下计算再转回）

---

## 11.2 SwiGLU vs GELU

### 理论

GPT-2 的 FFN 结构：
$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 \cdot x)$$

现代模型使用 **SwiGLU**（Swish-Gated Linear Unit）：
$$\text{SwiGLU}(x) = (\text{Swish}(x W_{gate}) \odot (x W_{up})) W_{down}$$

其中 Swish（也叫 SiLU）定义为：
$$\text{Swish}(x) = x \cdot \sigma(\beta x) = x \cdot \text{sigmoid}(x)$$

**GLU 家族**：GLU (Gated Linear Unit) 的核心思想是用一个「门控」分支来筛选信息。
- 门控信号决定哪些信息通过、哪些被抑制
- SwiGLU 用 Swish 激活作为门控，比 ReLU/GLU 效果更好

**三矩阵结构**：
- $W_{gate}$：门控投影（gate），决定信息通过的程度
- $W_{up}$：上投影（up），将维度从 $d$ 扩展到 $4d$
- $W_{down}$：下投影（down），将维度从 $4d$ 压回 $d$

参考论文：[GLU Variants Improve Transformer](https://arxiv.org/abs/2002.05202) (Shazeer, 2020)

In [ ]:
# ============================================================
# 手写 SwiGLU FFN（替代 GPT-2 的 GELU MLP）
# ============================================================

class SwiGLUFFN(nn.Module):
    """SwiGLU 前馈网络，替代 GPT-2 的 GELU MLP
    
    结构：x -> W_gate -> SiLU -> * (逐元素乘) -> W_down -> output
                    x -> W_up   -^
    
    三个矩阵：W_gate（门控）、W_up（上投影）、W_down（下投影）
    """
    def __init__(self, n_embd, hidden_dim=None, dropout=0.0):
        super().__init__()
        if hidden_dim is None:
            # Llama 风格：2/3 * 4 * n_embd，对齐到 256 的倍数
            hidden_dim = int(2 / 3 * 4 * n_embd)
            hidden_dim = ((hidden_dim + 255) // 256) * 256
        self.w_gate = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w_up   = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU = SiLU(gate) * up
        gate = self.w_gate(x)                    # (B, T, hidden_dim)
        up   = self.w_up(x)                      # (B, T, hidden_dim)
        y = F.silu(gate) * up                    # Swish(gate) * up <-- 门控机制
        y = self.w_down(y)                       # (B, T, n_embd)
        return self.dropout(y)


class GELU_MLP(nn.Module):
    """GPT-2 原始 MLP，用于对比
    
    结构：c_fc (n_embd -> 4*n_embd) -> GELU -> c_proj (4*n_embd -> n_embd)
    """
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return self.dropout(x)

In [ ]:
# ============================================================
# 对比 SwiGLU FFN vs GELU MLP：参数量、激活函数可视化
# ============================================================

torch.manual_seed(42)
n_embd = 128
x = torch.randn(2, 32, n_embd)

gelu_mlp = GELU_MLP(n_embd)
swiglu_ffn = SwiGLUFFN(n_embd)

# 参数量对比
gelu_params = sum(p.numel() for p in gelu_mlp.parameters())
swiglu_params = sum(p.numel() for p in swiglu_ffn.parameters())
print("GELU MLP (2 个矩阵):")
print(f"  c_fc:   {n_embd} x {4*n_embd} = {n_embd * 4 * n_embd:,} 参数")
print(f"  c_proj: {4*n_embd} x {n_embd} = {4*n_embd * n_embd:,} 参数")
print(f"  合计: {gelu_params:,} 参数")

# Llama 风格 hidden_dim
hidden_dim = int(2 / 3 * 4 * n_embd)
hidden_dim = ((hidden_dim + 255) // 256) * 256
print(f"\nSwiGLU FFN (3 个矩阵，hidden_dim={hidden_dim}):")
print(f"  w_gate: {n_embd} x {hidden_dim} = {n_embd * hidden_dim:,} 参数")
print(f"  w_up:   {n_embd} x {hidden_dim} = {n_embd * hidden_dim:,} 参数")
print(f"  w_down: {hidden_dim} x {n_embd} = {hidden_dim * n_embd:,} 参数")
print(f"  合计: {swiglu_params:,} 参数")
print(f"\nSwiGLU 参数量是 GELU MLP 的 {swiglu_params/gelu_params:.2f}x")

# 功能验证
y_gelu = gelu_mlp(x)
y_swiglu = swiglu_ffn(x)
print(f"\nGELU MLP  输出 shape: {y_gelu.shape}")
print(f"SwiGLU FFN 输出 shape: {y_swiglu.shape}")

# 可视化激活函数对比
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

t = torch.linspace(-4, 4, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# GELU
axes[0].plot(t.numpy(), F.gelu(t).numpy())
axes[0].set_title('GELU')
axes[0].grid(True, alpha=0.3)

# Swish / SiLU
axes[1].plot(t.numpy(), F.silu(t).numpy())
axes[1].set_title('Swish / SiLU')
axes[1].grid(True, alpha=0.3)

# 对比
axes[2].plot(t.numpy(), F.gelu(t).numpy(), label='GELU')
axes[2].plot(t.numpy(), F.silu(t).numpy(), label='Swish')
axes[2].set_title('GELU vs Swish')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('activation_comparison.png', dpi=100)
print("\n激活函数对比图已保存到 activation_comparison.png")
plt.show()

### 分析

- GELU 和 Swish 的曲线非常相似，都在负值区域有轻微的非零输出，在正值区域近似线性
- SwiGLU 的关键不是激活函数本身，而是**门控机制**：`silu(gate) * up` 让网络可以自适应地选择传递哪些信息
- 参数量代价：SwiGLU 需要 3 个矩阵而非 2 个，但 Llama 通过缩小 hidden_dim（$\frac{8}{3}d$ 对齐到 256 的倍数）来平衡参数量
- Shazeer (2020) 的实验表明，SwiGLU 在多个基准上优于其他 GLU 变体（ReGLU、GeGLU 等）

---

## 11.3 RoPE（旋转位置编码）

### 理论

GPT-2 使用**可学习的绝对位置编码**：为每个位置 $0, 1, ..., T-1$ 学习一个向量。

RoPE（Rotary Position Embedding）的核心思想：**将位置信息编码为旋转操作**。

**直觉理解**：
- 把 Q 和 K 的每对相邻维度看作二维平面上的点 $(x_{2i}, x_{2i+1})$
- 根据位置 $m$ 对这个点做旋转：角度为 $m \cdot \theta_i$
- 不同维度用不同的旋转频率 $\theta_i = 10000^{-2i/d}$

**数学原理**：

对于位置 $m$ 的向量 $x$，RoPE 将其变换为：

$$\begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix} \rightarrow \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

**关键性质**：
$$q_m^T k_n = \sum_i R_{m\theta_i} q_i \cdot R_{n\theta_i} k_i = \sum_i R_{(m-n)\theta_i} (q_i \cdot k_i)$$

内积只依赖于**相对位置** $m - n$！这就是 RoPE 能捕获相对位置信息的数学原因。

**优势**：
1. 自然编码相对位置关系
2. 可外推到更长序列（通过 NTK-aware scaling 等技术）
3. 不需要额外的位置 embedding 参数表

参考论文：[RoFormer](https://arxiv.org/abs/2104.09864) (Su et al., 2021)

In [ ]:
# ============================================================
# 手写 RoPE（~30 行）
# ============================================================

def precompute_freqs_cis(dim, max_seq_len, theta=10000.0):
    """预计算 RoPE 的旋转频率（复数形式）
    
    对于 dim 维的向量（必须是偶数），计算每对维度的旋转角度。
    freqs[i] = 1 / (theta ^ (2i / dim))，对应第 i 对维度的基础频率。
    
    参数:
        dim: head_dim（每个注意力头的维度）
        max_seq_len: 最大序列长度
        theta: 基础频率（默认 10000，Llama-3 用 500000）
    
    返回: (max_seq_len, dim/2) 的复数张量
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))  # (dim/2,)
    t = torch.arange(max_seq_len)  # (max_seq_len,)
    # 外积: 每个位置 m 对每个频率 i 的角度 m * freqs[i]
    freqs = torch.outer(t, freqs)  # (max_seq_len, dim/2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # 复数形式 e^(i*angle)
    return freqs_cis


def apply_rotary_emb(xq, xk, freqs_cis):
    """对 Q 和 K 应用 RoPE
    
    参数:
        xq, xk: (batch, seq_len, n_head, head_dim)
        freqs_cis: (seq_len, head_dim/2)
    
    步骤：
    1. 将相邻维度配对，视为复数
    2. 与预计算的旋转因子相乘（复数乘法 = 旋转）
    3. 将复数变回实数
    """
    # 将实数向量 reshape 成复数形式
    # (B, T, n_head, head_dim) -> (B, T, n_head, head_dim/2, 2) -> 复数
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # 广播并做复数乘法（旋转）
    # freqs_cis: (T, head_dim/2) -> (1, T, 1, head_dim/2)
    freqs_cis = freqs_cis[None, :xq_.shape[1], None, :]
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(-2)  # 复数 -> 实数
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(-2)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)

In [ ]:
# ============================================================
# 验证 RoPE 的相对位置性质 + 与绝对位置编码对比
# ============================================================

torch.manual_seed(42)
n_embd = 128
n_head = 4
head_dim = n_embd // n_head  # 32
seq_len = 16

# 生成 Q, K
q = torch.randn(1, seq_len, n_head, head_dim)
k = torch.randn(1, seq_len, n_head, head_dim)

# 预计算旋转频率
freqs_cis = precompute_freqs_cis(head_dim, seq_len)
print(f"freqs_cis shape: {freqs_cis.shape}")  # (seq_len, head_dim/2)

# 应用 RoPE
q_rot, k_rot = apply_rotary_emb(q, k, freqs_cis)
print(f"旋转后 Q shape: {q_rot.shape}")  # (1, seq_len, n_head, head_dim)

# 对比绝对位置编码
abs_pe = torch.randn(seq_len, n_embd)  # 可学习位置编码
q_flat = q.reshape(1, seq_len, n_embd)
k_flat = k.reshape(1, seq_len, n_embd)
q_abs = q_flat + abs_pe[None, :, :]
k_abs = k_flat + abs_pe[None, :, :]

# 计算 attention score
# RoPE
q_r = q_rot.transpose(1, 2)  # (1, n_head, T, head_dim)
k_r = k_rot.transpose(1, 2)
attn_rope = (q_r @ k_r.transpose(-2, -1))[0, 0] / math.sqrt(head_dim)

# 绝对 PE
attn_abs = (q_abs @ k_abs.transpose(-1, -2))[0] / math.sqrt(n_embd)

print("\nRoPE attention score 矩阵 (head 0, 前 6x6):")
print(attn_rope[:6, :6].detach().round(decimals=2))

# 检查对角线元素是否相等（相对位置为 0）
diag = attn_rope.diagonal()
print(f"\n对角线元素 (相对位置=0): {diag[:5].tolist()}")
print(f"对角线方差: {diag.var().item():.6f} (应接近 0，因为相对位置相同)")

# 可视化 RoPE vs 无位置编码
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 无位置编码
q_no_pe = q.transpose(1, 2)
k_no_pe = k.transpose(1, 2)
attn_no = F.softmax((q_no_pe @ k_no_pe.transpose(-2, -1))[0, 0] / math.sqrt(head_dim), dim=-1)
axes[0].imshow(attn_no.detach().numpy(), cmap='Blues')
axes[0].set_title('无位置编码')

# RoPE
attn_rp = F.softmax(attn_rope, dim=-1)
axes[1].imshow(attn_rp.detach().numpy(), cmap='Blues')
axes[1].set_title('RoPE')

plt.suptitle('注意力分布对比: RoPE 引入位置感知')
plt.tight_layout()
plt.savefig('rope_comparison.png', dpi=100)
print("\nRoPE 对比图已保存到 rope_comparison.png")
plt.show()

### 分析

- RoPE 通过复数旋转巧妙地将位置信息融入 Q 和 K
- 内积 $q_m^T k_n$ 自然地编码了相对位置 $m-n$，无需显式计算
- 与绝对位置编码不同，RoPE 不需要额外的 embedding 参数表
- 对角线元素（相对位置为 0）几乎完全一致，验证了 RoPE 的等变性
- RoPE 的另一个优势是**长度外推**：通过修改 $\theta$（如 NTK-aware scaling），可以将训练在短序列上的模型应用到长序列
- Llama-3 将 $\theta$ 从 10000 提高到 500000，以支持 8K+ 的上下文长度

---

## 11.4 GQA（分组查询注意力）

### 理论

注意力机制的演进：

| 类型 | Q heads | K/V heads | KV cache | 质量 |
|------|---------|-----------|----------|------|
| MHA (GPT-2) | n_head | n_head | 大 | 最高 |
| GQA (Llama-2) | n_head | n_kv_heads | 中 | 高 |
| MQA (PaLM) | n_head | 1 | 最小 | 较低 |

**MHA**：每个 Q head 有独立的 K 和 V head
**MQA**：所有 Q head 共享 1 组 K 和 V
**GQA**：将 Q heads 分成 `n_head / n_kv_heads` 组，每组共享一组 K/V

**KV cache 节省计算**：
- 生成时需要缓存每层的 K 和 V
- KV cache 大小 = $2 \times n\_layer \times n\_kv\_heads \times head\_dim \times seq\_len \times \text{bytes}$
- 减少 K/V heads 直接减少显存占用

参考论文：[GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245) (Ainslie et al., 2023)

In [ ]:
# ============================================================
# 手写 GQA (Grouped Query Attention)
# ============================================================

class GroupedQueryAttention(nn.Module):
    """分组查询注意力
    
    n_heads: Q 的头数
    n_kv_heads: K/V 的头数（<= n_heads）
    
    当 n_kv_heads == n_heads 时，等价于 MHA
    当 n_kv_heads == 1 时，等价于 MQA
    """
    def __init__(self, n_embd, n_heads, n_kv_heads, block_size, dropout=0.0):
        super().__init__()
        assert n_embd % n_heads == 0
        assert n_heads % n_kv_heads == 0  # Q heads 必须能被 KV heads 整除
        
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = n_embd // n_heads
        self.n_rep = n_heads // n_kv_heads  # 每个 KV head 对应几个 Q head
        
        # Q: n_heads 个头, K/V: n_kv_heads 个头
        self.wq = nn.Linear(n_embd, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(n_embd, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(n_embd, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, n_embd, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
        # causal mask
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def _repeat_kv(self, x):
        """将 KV heads 重复以匹配 Q heads 的数量
        
        x: (B, n_kv_heads, T, head_dim) -> (B, n_heads, T, head_dim)
        """
        if self.n_rep == 1:
            return x  # MHA，不需要重复
        B, n_kv, T, D = x.shape
        x = x[:, :, None, :, :].expand(B, n_kv, self.n_rep, T, D)
        return x.reshape(B, n_kv * self.n_rep, T, D)

    def forward(self, x, freqs_cis=None):
        B, T, C = x.size()
        
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)      # (B, T, n_heads, head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)   # (B, T, n_kv_heads, head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)   # (B, T, n_kv_heads, head_dim)
        
        # 应用 RoPE（如果提供了旋转频率）
        if freqs_cis is not None:
            q, k = apply_rotary_emb(q, k, freqs_cis)
        
        # 转置为 (B, heads, T, head_dim) 便于矩阵乘法
        q = q.transpose(1, 2)   # (B, n_heads, T, head_dim)
        k = k.transpose(1, 2)   # (B, n_kv_heads, T, head_dim)
        v = v.transpose(1, 2)   # (B, n_kv_heads, T, head_dim)
        
        # 重复 KV heads 以匹配 Q heads
        k = self._repeat_kv(k)  # (B, n_heads, T, head_dim)
        v = self._repeat_kv(v)  # (B, n_heads, T, head_dim)
        
        # Scaled dot-product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        y = att @ v  # (B, n_heads, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, n_embd)
        y = self.resid_dropout(self.wo(y))
        return y

In [ ]:
# ============================================================
# KV cache 大小对比：MHA vs GQA vs MQA（具体数字）
# ============================================================

# 模拟 Llama-2 70B 级别的配置
configs = {
    "MHA (GPT-2 风格)": {"n_layer": 80, "n_heads": 64, "n_kv_heads": 64, "head_dim": 128},
    "GQA-8 (Llama-2 70B)": {"n_layer": 80, "n_heads": 64, "n_kv_heads": 8, "head_dim": 128},
    "GQA-4": {"n_layer": 80, "n_heads": 64, "n_kv_heads": 4, "head_dim": 128},
    "MQA (n_kv=1)": {"n_layer": 80, "n_heads": 64, "n_kv_heads": 1, "head_dim": 128},
}

seq_len = 4096
bytes_per_element = 2  # float16

print(f"序列长度: {seq_len}, 精度: float16, 层数: 80")
print(f"\n{'配置':25s} {'KV heads':>10s} {'KV cache (MB)':>15s} {'节省比例':>10s}")
print("-" * 65)

mha_cache = None
for name, cfg in configs.items():
    # KV cache = 2 (K+V) * n_layer * n_kv_heads * head_dim * seq_len * bytes
    cache_bytes = (2 * cfg["n_layer"] * cfg["n_kv_heads"] * cfg["head_dim"] 
                   * seq_len * bytes_per_element)
    cache_mb = cache_bytes / (1024 ** 2)
    
    if mha_cache is None:
        mha_cache = cache_mb
        saving = "-"
    else:
        saving = f"{(1 - cache_mb/mha_cache)*100:.0f}%"
    
    print(f"{name:25s} {cfg['n_kv_heads']:>10d} {cache_mb:>15.0f} {saving:>10s}")

print(f"\n结论: GQA-8 相比 MHA 节省了 {(1 - 8/64)*100:.0f}% 的 KV cache 显存")

In [ ]:
# ============================================================
# 验证 GQA 正确性：不同 KV head 数的 forward pass
# ============================================================

torch.manual_seed(42)
n_embd = 128
n_head = 4
block_size = 64
x = torch.randn(2, 32, n_embd)

print("测试不同 KV head 数的 GQA:\n")

# MHA: n_kv_heads == n_heads
attn_mha = GroupedQueryAttention(n_embd, n_head, n_kv_heads=4, block_size=block_size)
y_mha = attn_mha(x)
params_mha = sum(p.numel() for p in attn_mha.parameters())
print(f"MHA  (n_kv=4): output={y_mha.shape}, params={params_mha:,}")

# GQA-2: n_kv_heads = 2
attn_gqa2 = GroupedQueryAttention(n_embd, n_head, n_kv_heads=2, block_size=block_size)
y_gqa2 = attn_gqa2(x)
params_gqa2 = sum(p.numel() for p in attn_gqa2.parameters())
print(f"GQA-2 (n_kv=2): output={y_gqa2.shape}, params={params_gqa2:,}")

# MQA: n_kv_heads = 1
attn_mqa = GroupedQueryAttention(n_embd, n_head, n_kv_heads=1, block_size=block_size)
y_mqa = attn_mqa(x)
params_mqa = sum(p.numel() for p in attn_mqa.parameters())
print(f"MQA  (n_kv=1): output={y_mqa.shape}, params={params_mqa:,}")

print(f"\n参数量比例:")
print(f"  MHA  : GQA-2 : MQA = {params_mha} : {params_gqa2} : {params_mqa}")
print(f"  即   : {params_mha/params_mha:.2f}x : {params_gqa2/params_mha:.2f}x : {params_mqa/params_mha:.2f}x")

### 分析

- GQA 是 MHA 和 MQA 的折中：保留大部分质量，同时显著减少 KV cache
- 参数减少主要来自 K/V 的投影矩阵（`wk` 和 `wv` 的输出维度从 `n_heads * head_dim` 降到 `n_kv_heads * head_dim`）
- Llama-2 70B 使用 GQA-8（64 个 Q heads / 8 个 KV heads），KV cache 节省 87.5%
- Llama-3 进一步使用了 GQA-8，而 Mistral-7B 使用 GQA-1（即 MQA）
- Ainslie et al. (2023) 的实验表明 GQA 在质量和推理速度之间取得了最佳平衡

---

## 11.5 集成：组装 ModernGPT

现在我们将四个改进组合到一个完整的模型中：

| 改进 | 替换对象 | 效果 |
|------|---------|------|
| RMSNorm | LayerNorm | 更快、更少参数 |
| SwiGLU | GELU MLP | 更好的 FFN 效果 |
| RoPE | 绝对位置编码 | 相对位置 + 可外推 |
| GQA | MHA | 更小的 KV cache |

In [ ]:
# ============================================================
# ModernGPT: 集成四大改进的完整 Transformer
# ============================================================

class ModernGPTConfig:
    """现代 GPT 配置"""
    def __init__(self, vocab_size=32000, block_size=2048,
                 n_layer=12, n_head=12, n_embd=768,
                 n_kv_heads=None,   # None 表示等于 n_head (MHA)
                 dropout=0.0,
                 rope_theta=10000.0):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.n_kv_heads = n_kv_heads if n_kv_heads is not None else n_head
        self.dropout = dropout
        self.rope_theta = rope_theta


class ModernBlock(nn.Module):
    """现代 Transformer Block
    
    集成：RMSNorm + GQA(+RoPE) + SwiGLU
    """
    def __init__(self, config):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)                        # 改进 1: RMSNorm
        self.attn = GroupedQueryAttention(                         # 改进 3+4: RoPE + GQA
            config.n_embd, config.n_head, config.n_kv_heads, 
            config.block_size, config.dropout
        )
        self.ln_2 = RMSNorm(config.n_embd)                        # 改进 1: RMSNorm
        self.ffn = SwiGLUFFN(config.n_embd, dropout=config.dropout)  # 改进 2: SwiGLU

    def forward(self, x, freqs_cis):
        x = x + self.attn(self.ln_1(x), freqs_cis)  # Pre-norm + 注意力
        x = x + self.ffn(self.ln_2(x))               # Pre-norm + FFN
        return x


class ModernGPT(nn.Module):
    """集成四大改进的现代 GPT 模型
    
    架构对比 GPT-2:
    - LayerNorm -> RMSNorm
    - GELU MLP -> SwiGLU FFN  
    - 可学习绝对位置编码 -> RoPE
    - MHA -> GQA
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.block_size = config.block_size
        
        # Token embedding（无位置编码！RoPE 在 attention 内部处理位置）
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            ModernBlock(config) for _ in range(config.n_layer)
        ])
        
        # 最终归一化
        self.norm = RMSNorm(config.n_embd)
        
        # Output head
        self.output = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Weight tying
        self.tok_embeddings.weight = self.output.weight
        
        # 预计算 RoPE 频率（不需要梯度）
        head_dim = config.n_embd // config.n_head
        self.register_buffer(
            "freqs_cis",
            precompute_freqs_cis(head_dim, config.block_size, config.rope_theta)
        )
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size, f"序列长度 {T} 超过最大长度 {self.block_size}"
        
        x = self.tok_embeddings(idx)  # (B, T, n_embd) -- 无位置编码！
        x = self.dropout(x)
        
        freqs_cis = self.freqs_cis[:T]  # 取前 T 个位置的旋转频率
        
        for layer in self.layers:
            x = layer(x, freqs_cis)
        
        x = self.norm(x)
        logits = self.output(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss

In [ ]:
# ============================================================
# GPT-2 风格 vs Modern 风格：完整参数量对比
# ============================================================

# Baby 配置，CPU 可跑
vocab_size = 32000
block_size = 256
n_layer = 4
n_head = 4
n_embd = 128

# --- 构建 GPT-2 风格模型（复用第1章定义） ---
# 需要第1章的 GPT, GPTConfig, CausalSelfAttention, MLP, Block
# 这里用参数量公式直接计算

def count_gpt2_params(vocab_size, block_size, n_layer, n_head, n_embd):
    """计算 GPT-2 风格模型的参数量"""
    V, D, L = vocab_size, n_embd, n_layer
    emb = V * D                          # token embedding
    pe = block_size * D                  # position embedding
    per_block = (
        3 * D * D + 3 * D +              # c_attn (QKV 合并, 含 bias)
        D * D + D +                      # c_proj (含 bias)
        2 * (D + D) +                    # 2 x LayerNorm (weight + bias)
        D * 4*D + 4*D +                  # c_fc (含 bias)
        4*D * D + D                      # c_proj (含 bias)
    )
    ln_f = 2 * D                         # final LayerNorm
    head = 0                             # weight tying, 不重复计算
    return emb + pe + L * per_block + ln_f + head

def count_modern_params(vocab_size, block_size, n_layer, n_head, n_kv_heads, n_embd):
    """计算 ModernGPT 的参数量"""
    V, D, L = vocab_size, n_embd, n_layer
    head_dim = D // n_head
    emb = V * D                          # token embedding
    # RoPE: 0 可学习参数
    hidden_dim = int(2/3 * 4 * D)
    hidden_dim = ((hidden_dim + 255) // 256) * 256
    per_block = (
        n_head * head_dim * D +          # wq (无 bias)
        n_kv_heads * head_dim * D +      # wk (无 bias)
        n_kv_heads * head_dim * D +      # wv (无 bias)
        D * D +                          # wo (无 bias)
        2 * D +                          # 2 x RMSNorm (weight only)
        hidden_dim * D * 3               # SwiGLU: w_gate + w_up + w_down (无 bias)
    )
    norm_f = D                           # final RMSNorm
    head = 0                             # weight tying
    return emb + L * per_block + norm_f + head

gpt2_total = count_gpt2_params(vocab_size, block_size, n_layer, n_head, n_embd)
modern_gqa4 = count_modern_params(vocab_size, block_size, n_layer, n_head, 4, n_embd)
modern_gqa2 = count_modern_params(vocab_size, block_size, n_layer, n_head, 2, n_embd)
modern_gqa1 = count_modern_params(vocab_size, block_size, n_layer, n_head, 1, n_embd)

print("=" * 60)
print(f"GPT-2 风格 vs Modern 风格 参数量对比")
print(f"配置: {n_layer} 层, {n_head} heads, {n_embd} embd, vocab={vocab_size}")
print("=" * 60)
print(f"\n{'模型':30s} {'参数量':>12s} {'比值':>8s}")
print("-" * 55)
print(f"{'GPT-2 风格 (MHA, LN, GELU, AbsPE)':30s} {gpt2_total:>12,} {'1.00x':>8s}")
print(f"{'Modern (GQA-4=MHA, RMSNorm, SwiGLU, RoPE)':30s} {modern_gqa4:>12,} {f'{modern_gqa4/gpt2_total:.2f}x':>8s}")
print(f"{'Modern (GQA-2, RMSNorm, SwiGLU, RoPE)':30s} {modern_gqa2:>12,} {f'{modern_gqa2/gpt2_total:.2f}x':>8s}")
print(f"{'Modern (MQA, RMSNorm, SwiGLU, RoPE)':30s} {modern_gqa1:>12,} {f'{modern_gqa1/gpt2_total:.2f}x':>8s}")

print(f"\n参数量差异分析:")
print(f"  - RoPE 省去位置编码: -{block_size * n_embd:,} 参数")
print(f"  - RMSNorm 省去 bias: 每层 -{2 * n_embd} 参数 x {n_layer} 层")
print(f"  - GQA-2 减少 K/V 投影: 每层 -{2 * (n_head - 2) * (n_embd // n_head) * n_embd:,} 参数 x {n_layer} 层")
print(f"  - SwiGLU 多一个矩阵: 每层 +{hidden_dim * n_embd:,} 参数 x {n_layer} 层")

In [ ]:
# ============================================================
# Forward pass 测试：验证 ModernGPT 可正常训练
# ============================================================

torch.manual_seed(42)

# Baby 配置
config = ModernGPTConfig(
    vocab_size=65, block_size=128,
    n_layer=4, n_head=4, n_embd=128,
    n_kv_heads=2,  # GQA-2
)

model = ModernGPT(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"ModernGPT 参数量: {total_params:,} ({total_params/1e6:.2f}M)")

# Forward pass
x = torch.randint(0, config.vocab_size, (2, 64))  # batch=2, seq_len=64
logits, loss = model(x, targets=x)
print(f"logits shape: {logits.shape}")
print(f"loss: {loss.item():.4f}")

# 简单训练 10 步，验证 loss 下降
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print(f"\n训练 10 步:")
for step in range(10):
    optimizer.zero_grad()
    logits, loss = model(x, targets=x)
    loss.backward()
    optimizer.step()
    if step % 2 == 0:
        print(f"  step {step}: loss = {loss.item():.4f}")

print("\nModernGPT 训练成功！loss 应在下降。")

In [ ]:
# ============================================================
# 逐层参数分解：GPT-2 vs Modern
# ============================================================

n_embd_demo = 128
n_head_demo = 4
n_kv_demo = 2

# GPT-2 单层
n = n_embd_demo
gpt2_attn = 3*n*n + 3*n + n*n + n     # c_attn + c_proj (含 bias)
gpt2_mlp = n*4*n + 4*n + 4*n*n + n     # c_fc + c_proj (含 bias)
gpt2_norm = 2 * (n + n)                 # 2 x LayerNorm
gpt2_total_layer = gpt2_attn + gpt2_mlp + gpt2_norm

# Modern 单层
hd = n // n_head_demo  # head_dim = 32
hidden_dim = int(2/3 * 4 * n)
hidden_dim = ((hidden_dim + 255) // 256) * 256
mod_attn = n_head_demo*hd*n + n_kv_demo*hd*n + n_kv_demo*hd*n + n*n  # wq + wk + wv + wo (无 bias)
mod_ffn = hidden_dim * n * 3             # w_gate + w_up + w_down (无 bias)
mod_norm = 2 * n                         # 2 x RMSNorm
mod_total_layer = mod_attn + mod_ffn + mod_norm

print(f"单层参数分解 (n_embd={n_embd_demo}, n_head={n_head_demo}):")
print(f"\n{'组件':20s} {'GPT-2':>12s} {'Modern':>12s} {'说明':>30s}")
print("-" * 80)
print(f"{'Attention':20s} {gpt2_attn:>12,} {mod_attn:>12,} {'MHA vs GQA-2':>30s}")
print(f"{'FFN / MLP':20s} {gpt2_mlp:>12,} {mod_ffn:>12,} {'GELU 2-矩阵 vs SwiGLU 3-矩阵':>30s}")
print(f"{'归一化':20s} {gpt2_norm:>12,} {mod_norm:>12,} {'LayerNorm vs RMSNorm':>30s}")
print("-" * 80)
print(f"{'单层合计':20s} {gpt2_total_layer:>12,} {mod_total_layer:>12,} {f'{mod_total_layer/gpt2_total_layer:.2f}x':>30s}")

### 集成分析

**参数量变化总结**：
- **RMSNorm 减少**：每层省 $n$ 个参数（去掉 bias），总计 $2 \times n \times L$
- **GQA 减少**：K/V 投影矩阵缩小为原来的 `n_kv_heads / n_heads`
- **RoPE 减少**：去掉位置编码参数表（$T \times n_{embd}$）
- **SwiGLU 增加**：多一个投影矩阵，FFN 参数约 1.5x（但 hidden_dim 缩小后约 1.0-1.2x）

**综合来看**：Modern 风格的参数量与 GPT-2 风格相近，但在以下方面有显著提升：
1. 推理效率更高（更小的 KV cache、更快的归一化）
2. 训练更稳定（RMSNorm + SwiGLU 门控）
3. 位置编码更灵活（RoPE 支持长度外推）

**生产级参考实现**：HuggingFace `transformers` 库的 [LlamaModel](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py) 是这些改进的最佳参考。

---

## 论文参考

| 改进 | 论文 | 链接 |
|------|------|------|
| RMSNorm | Root Mean Square Layer Normalization (Zhang & Sennrich, 2019) | https://arxiv.org/abs/1910.07467 |
| SwiGLU | GLU Variants Improve Transformer (Shazeer, 2020) | https://arxiv.org/abs/2002.05202 |
| RoPE | RoFormer: Enhanced Transformer with Rotary Position Embedding (Su et al., 2021) | https://arxiv.org/abs/2104.09864 |
| GQA | GQA: Training Generalized Multi-Query Transformer (Ainslie et al., 2023) | https://arxiv.org/abs/2305.13245 |
| 参考 | HuggingFace Llama 源码 | https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py |

## 练习

1. **RMSNorm 实验**：修改 `eps` 值（1e-5, 1e-6, 1e-8），观察输出变化。为什么通常用 1e-6？

2. **SwiGLU 变体**：实现其他 GLU 变体（ReGLU、GeGLU），对比它们与 SwiGLU 的差异。
   - ReGLU: `ReLU(gate) * up`
   - GeGLU: `GELU(gate) * up`

3. **RoPE 长度外推**：用 NTK-aware scaling 修改 `rope_theta`，让训练在 `block_size=256` 的模型能处理 `block_size=512` 的序列。
   - 提示：新的 `theta' = theta * (scale_factor)^(dim/(dim-2))`，其中 `scale_factor = new_length / old_length`

4. **GQA 消融**：用相同的随机种子训练 baby-GPT 和 baby-ModernGPT（用 Shakespeare 数据集），对比两者的训练 loss 曲线。

5. **阅读源码**：打开 [HuggingFace Llama modeling_llama.py](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)，找到本章实现的每个组件对应的源码位置。注意 Llama 在哪些地方做了额外的优化（如 `sdpa`、`flash_attention`）。